In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Read the TSV files into the DataFrame
#Data Source Hasso Plattner Institut 
# NDPL - Non-duplicates
# DPL - Duplicates
# ncvoters - a snap shot of the snapshot: VR_Snapshot_20181106 
df_ncvoters = pd.read_csv('/Users/noimotbakare/Dropbox/Mac/Downloads/ncvoters.tsv', sep='\t')
DPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_DPL.tsv', sep='\t')
NDPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_NDPL.tsv', sep='\t')



In [ ]:
# Preprocessing for our fragemented ID analysis 
# Selecting identifyer variables not related to voting
df_ncvoters_frag_ID = df_ncvoters[[
#Stable Identifiers
#Only to be used for labeling /evaluation only 
# / not as a model feature
    'id', 'ncid', 'voter_reg_num', 
# Primary Model features - Strongest features, Strong entropy, Essential for matching
    #many variable such as name prefx and sufx are sparse we can either use none for missing or 0/1
     'first_name', 'midl_name', 'last_name', 'name_sufx_cd',
    #other varaiabes
# Adress Similarity features - Address is the second strongest identity anchor, 
    # Street name especially high discriminative signal # Unit numbers distinguish household
    #many variable such as unit designator are sparse we can either use none for missing or 0/1
    'house_num', 'street_name', 'street_dir', 'street_type_cd', 'unit_designator', 'unit_num', 'zip_code', 'res_city_desc',
#other varaiabes
# Demographic agreement indicators - Moderate/Supporting Features 
    # these will help us reduce false matches # they are agreement indicators, low-weight similarity features
    #age group rather than age because grouped/bin age is more stable
    'age', 'age_group', 'sex', 'race_code', 'race_desc', 'ethnic_code', 'ethnic_desc', 'birth_place',
# #other varaiabes 
 'phone_num','area_cd'   
 ]]
# print("\nSelected variables 'A' and 'C':")
print(df_ncvoters_frag_ID)

In [ ]:
# Adding labels to DPL and NDPL
DPL['label'] = 1 
NDPL['label'] = 0


print(DPL.head(10))
print(NDPL.head(10))

In [ ]:
pairs = pd.concat([DPL, NDPL])

print(pairs.head(20))
print(pairs.tail(20))

In [ ]:
# merging ncvoters on DPL and NDPL
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id1",
    right_on="id",
    how="left"
)
print(pairs.info())
print(pairs.head(10))

In [ ]:
# merging id2 
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id2",
    right_on="id",
    how="left",
    suffixes=("_1", "_2")
)

print(pairs.info())
print(pairs.head(10))

Siamese Network Features

In [ ]:
# ========== SIAMESE NETWORK FEATURES ==========
# Raw text fields that will be embedded
siamese_features = [
    # Names
    'first_name_1', 'first_name_2',
    'midl_name_1', 'midl_name_2', 
    'last_name_1', 'last_name_2',
    'name_sufx_cd_1', 'name_sufx_cd_2',
    
    # Address
    'house_num_1', 'house_num_2',
    'street_name_1', 'street_name_2',
    'street_type_cd_1', 'street_type_cd_2',
    'unit_num_1', 'unit_num_2',
    'zip_code_1', 'zip_code_2',
    'res_city_desc_1', 'res_city_desc_2',
    
    # Demographics
    'age_1', 'age_2',
    'phone_num_1', 'phone_num_2',
    'sex_1', 'sex_2',
    'race_desc_1', 'race_desc_2'
]
print(f"Siamese features: {len(siamese_features)}")

Baseline Model Features 

In [ ]:
# Same last name feature 
pairs["same_last_name"] = (
    pairs["last_name_1"].str.lower().fillna("") ==
    pairs["last_name_2"].str.lower().fillna("")
).astype(int)


In [ ]:
#Similarity feature generation

pairs["same_last_name"] = (
    pairs["last_name_1"].str.lower().fillna("") ==
    pairs["last_name_2"].str.lower().fillna("")
).astype(int)

pairs["age"] = (
    pairs["age_1"] ==
    pairs["age_2"]
).astype(int)

pairs["same_address"] = (
    pairs["house_num_1"] ==
    pairs["house_num_2"]
).astype(int)

pairs["same_street_name"] = (
    pairs["street_name_1"].str.lower().fillna("") ==
    pairs["street_name_2"].str.lower().fillna("")
).astype(int)

In [ ]:
#Overall Separability Check - Similarity Scores 
from rapidfuzz.fuzz import ratio

pairs["first_name_sim"] = pairs.apply(
    lambda x: ratio(
        str(x["first_name_1"]),
        str(x["first_name_2"])
    ),
    axis=1
)

pairs["last_name_sim"] = pairs.apply(
    lambda x: ratio(
        str(x["last_name_1"]),
        str(x["last_name_2"])
    ),
    axis=1
)

In [ ]:
exact_cols = [
    "first_name",
    "last_name",
    "midl_name",
    "house_num",
    "street_name",
    "zip_code",
    "res_city_desc",
    "sex",
    "race_desc",
    "ethnic_desc"
]

for col in exact_cols:
    pairs[f"{col}_exact"] = (
        pairs[f"{col}_1"] == pairs[f"{col}_2"]
    ).astype(int)

In [ ]:
pairs["age_diff"] = abs(pairs["age_1"] - pairs["age_2"])
pairs["age_exact"] = (pairs["age_diff"] == 0).astype(int)
pairs["age_close"] = (pairs["age_diff"] <= 1).astype(int)
pairs.head(50)

Creating binary columns for names that have the same phonetic pronounciation - Real duplicates often differ by spelling but not pronunciation. Phonetic match = strong evidence of same person.


In [ ]:
import jellyfish

def phonetic_features(pairs, col):

   pairs[f"{col}_soundex_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.soundex)
   pairs[f"{col}_soundex_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.soundex)

   pairs[f"{col}_metaphone_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.metaphone)
   pairs[f"{col}_metaphone_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.metaphone)

   pairs[f"{col}_soundex_match"] = (
        pairs[f"{col}_soundex_1"] == pairs[f"{col}_soundex_2"]
    ).astype(int)

   pairs[f"{col}_metaphone_match"] = (
        pairs[f"{col}_metaphone_1"] == pairs[f"{col}_metaphone_2"]
    ).astype(int)

   pairs.head(50)

In [ ]:
for col in ["first_name", "last_name"]:
    phonetic_features(pairs, col)

In [ ]:
# Missing agreement feature 
for col in ["midl_name", "phone_num"]:
    pairs[f"{col}_both_missing"] = (
        (pairs[f"{col}_1"] == "") &
        (pairs[f"{col}_2"] == "")
    ).astype(int)

In [ ]:

# ========== BASELINE MODEL FEATURES ==========
# Engineered comparison features for traditional ML
model_features = [
    # Exact matches
    'same_last_name', 'same_street_name',
    'first_name_exact', 'last_name_exact', 'midl_name_exact',
    'house_num_exact', 'street_name_exact', 'zip_code_exact',
    'res_city_desc_exact', 'sex_exact', 'race_desc_exact',
    # Similarity scores
    'first_name_sim', 'last_name_sim', 
    
    # Age comparisons
    'age_diff', 'age_exact', 'age_close',
    
    # Phonetic matches
    'first_name_soundex_match', 'first_name_metaphone_match',
    'last_name_soundex_match', 'last_name_metaphone_match',
    
    # Missingness indicators
    'midl_name_both_missing', 'phone_num_both_missing'
]

# ========== METADATA (for tracking, not training) ==========
metadata_cols = ['id1', 'id2', 'label', 'participation']

print(f"Baseline model features: {len(model_features)}")